# Step 3: Cohort Construction
## Building the Analytic Dataset from 10 NHANES Cycles (1999-2018)

**Objective:** Pool all 10 cycles, apply inclusion/exclusion criteria, harmonize variables, construct derived exposures and outcomes, and produce a single analysis-ready CSV.

---

### Inclusion Criteria
1. Age >= 20 at NHANES exam (standard adult definition)
2. Eligible for mortality follow-up (ELIGSTAT == 1)
3. Positive follow-up time (PERMTH_EXM > 0)
4. Valid cancer status (MCQ220 == 1 or 2; exclude refused/don't know)

### Variables to Construct
| Variable | Source | Definition |
|----------|--------|------------|
| cancer | MCQ220 | 1 = ever told had cancer |
| diabetes | DIQ010 + LBXGH | Doctor-told OR HbA1c >= 6.5% |
| hypertension | BPQ020 | 1 = ever told high BP |
| obese | BMXBMI | BMI >= 30 |
| bmi_cat | BMXBMI | Underweight/Normal/Overweight/Obese |
| smoking | SMQ020 + SMQ040 | Never/Former/Current |
| follow_up_yrs | PERMTH_EXM | Person-months / 12 |
| dead | MORTSTAT | 0 = alive, 1 = deceased |

In [1]:
# ============================================================
# STEP 3.1 — Setup
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
CYCLES = [
    '1999-2000', '2001-2002', '2003-2004', '2005-2006', '2007-2008',
    '2009-2010', '2011-2012', '2013-2014', '2015-2016', '2017-2018'
]

def load_mortality(cycle):
    """Parse NCHS fixed-width mortality file line by line.
    
    Format: SEQN (cols 0-13), ELIGSTAT (col 14), MORTSTAT (col 15).
    
    CRITICAL: PERMTH_INT and PERMTH_EXM can appear either:
      - Space-separated: '... 18 18' (2 separate tokens at end)
      - Concatenated: '...244244' (1 token, 6 digits = 244 + 244)
    We detect concatenation by checking if the last token is > 3 digits.
    Valid person-months range: 0-300 (25 years max follow-up through Dec 2019).
    """
    fpath = DATA_DIR / f'{cycle}_mortality.dat'
    rows = []
    with open(fpath) as f:
        for line in f:
            raw = line.rstrip()
            if len(raw) < 16:
                continue
            seqn = int(raw[0:14].strip())
            eligstat = float(raw[14]) if raw[14] != '.' else np.nan
            mortstat = float(raw[15]) if raw[15] != '.' else np.nan
            
            permth_int = np.nan
            permth_exm = np.nan
            
            parts = raw.split()
            
            # Strategy: look at the LAST token(s) for follow-up time
            # If last token is a number > 999, it's concatenated PERMTH_INT + PERMTH_EXM
            last = parts[-1] if parts else ''
            second_last = parts[-2] if len(parts) >= 2 else ''
            
            if last == '.' or last == '..':
                # No follow-up data
                pass
            elif last.isdigit() and len(last) > 3:
                # Concatenated: split in half
                mid = len(last) // 2
                permth_int = float(last[:mid])
                permth_exm = float(last[mid:])
            elif last.isdigit() and second_last.isdigit():
                # Separate tokens: second_last=PERMTH_INT, last=PERMTH_EXM
                permth_int = float(second_last)
                permth_exm = float(last)
            elif last.isdigit():
                # Only one number available
                permth_exm = float(last)
            
            rows.append({
                'SEQN': seqn,
                'ELIGSTAT': eligstat,
                'MORTSTAT': mortstat,
                'PERMTH_INT': permth_int,
                'PERMTH_EXM': permth_exm
            })
    
    return pd.DataFrame(rows)

# Validate across multiple cycles — follow-up should be 0-300 months (0-25 years)
for test_cycle in ['1999-2000', '2005-2006', '2011-2012', '2017-2018']:
    t = load_mortality(test_cycle)
    te = t[t['ELIGSTAT'] == 1]
    pm = te['PERMTH_EXM'].dropna()
    print(f'{test_cycle}: N={len(te):,}, Deaths={int((te["MORTSTAT"]==1).sum())}, '
          f'PERMTH_EXM range={pm.min():.0f}-{pm.max():.0f} months ({pm.min()/12:.1f}-{pm.max()/12:.1f} yrs)')

print('\nSetup complete.')

1999-2000: N=5,445, Deaths=1675, PERMTH_EXM range=1-249 months (0.1-20.8 yrs)
2005-2006: N=5,561, Deaths=1027, PERMTH_EXM range=1-199 months (0.1-16.6 yrs)
2011-2012: N=5,849, Deaths=628, PERMTH_EXM range=0-199 months (0.0-16.6 yrs)
2017-2018: N=5,809, Deaths=145, PERMTH_EXM range=0-37 months (0.0-3.1 yrs)

Setup complete.


In [2]:
# ============================================================
# STEP 3.2 — Load and merge all 10 cycles
# ============================================================
# For each cycle: load 7 survey components + mortality,
# select key variables, merge on SEQN, tag with cycle label.

all_cycles = []

for cycle in CYCLES:
    # Load components
    demo = pd.read_sas(DATA_DIR / f'{cycle}_demo.XPT', format='xport')
    mcq  = pd.read_sas(DATA_DIR / f'{cycle}_mcq.XPT', format='xport')
    diq  = pd.read_sas(DATA_DIR / f'{cycle}_diq.XPT', format='xport')
    bmx  = pd.read_sas(DATA_DIR / f'{cycle}_bmx.XPT', format='xport')
    bpq  = pd.read_sas(DATA_DIR / f'{cycle}_bpq.XPT', format='xport')
    smq  = pd.read_sas(DATA_DIR / f'{cycle}_smq.XPT', format='xport')
    ghb  = pd.read_sas(DATA_DIR / f'{cycle}_ghb.XPT', format='xport')
    mort = load_mortality(cycle)
    
    # Select key variables (handle missing columns gracefully)
    def safe_select(df, cols):
        available = [c for c in cols if c in df.columns]
        return df[available].copy()
    
    d = safe_select(demo, ['SEQN','RIDAGEYR','RIAGENDR','RIDRETH1','DMDEDUC2','INDFMPIR','WTMEC2YR','SDMVPSU','SDMVSTRA'])
    m = safe_select(mcq, ['SEQN','MCQ220'])
    di = safe_select(diq, ['SEQN','DIQ010'])
    b = safe_select(bmx, ['SEQN','BMXBMI'])
    bp = safe_select(bpq, ['SEQN','BPQ020'])
    s = safe_select(smq, ['SEQN','SMQ020','SMQ040'])
    g = safe_select(ghb, ['SEQN','LBXGH'])
    mo = safe_select(mort, ['SEQN','ELIGSTAT','MORTSTAT','UCOD_LEADING','PERMTH_INT','PERMTH_EXM'])
    
    # Ensure SEQN is int everywhere
    for df in [d, m, di, b, bp, s, g, mo]:
        df['SEQN'] = df['SEQN'].astype(int)
    
    # Sequential merge
    merged = d.copy()
    for df in [m, di, b, bp, s, g, mo]:
        merged = merged.merge(df, on='SEQN', how='left')
    
    merged['cycle'] = cycle
    all_cycles.append(merged)
    print(f'{cycle}: {len(merged):,} rows merged')

raw = pd.concat(all_cycles, ignore_index=True)
print(f'\nPooled raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} cols')

1999-2000: 9,965 rows merged


2001-2002: 11,039 rows merged
2003-2004: 10,122 rows merged


2005-2006: 10,348 rows merged


2007-2008: 10,149 rows merged
2009-2010: 10,537 rows merged


2011-2012: 9,756 rows merged


2013-2014: 10,175 rows merged
2015-2016: 9,971 rows merged


2017-2018: 9,254 rows merged

Pooled raw dataset: 101,316 rows x 21 cols


In [3]:
# ============================================================
# STEP 3.3 — Apply inclusion/exclusion criteria
# ============================================================
# Track each exclusion step (CONSORT-style flow)

flow = []
flow.append(('Total pooled participants', len(raw)))

# 1. Adults >= 20
df = raw[raw['RIDAGEYR'] >= 20].copy()
flow.append(('Adults >= 20 years', len(df)))

# 2. Eligible for mortality follow-up
df = df[df['ELIGSTAT'] == 1].copy()
flow.append(('Eligible for mortality linkage', len(df)))

# 3. Positive follow-up time
df = df[df['PERMTH_EXM'] > 0].copy()
flow.append(('Positive follow-up time', len(df)))

# 4. Valid cancer status (MCQ220 == 1 or 2)
df = df[df['MCQ220'].isin([1.0, 2.0])].copy()
flow.append(('Valid cancer status (yes/no)', len(df)))

# 5. Non-missing BMI (needed for obesity exposure)
df = df[df['BMXBMI'].notna()].copy()
flow.append(('Non-missing BMI', len(df)))

print('=== CONSORT-STYLE EXCLUSION FLOW ===')
for i, (label, n) in enumerate(flow):
    excluded = flow[i-1][1] - n if i > 0 else 0
    exc_str = f'  (excluded {excluded:,})' if excluded > 0 else ''
    print(f'  {label}: {n:,}{exc_str}')

print(f'\nFinal analytic sample: {len(df):,}')

=== CONSORT-STYLE EXCLUSION FLOW ===
  Total pooled participants: 101,316
  Adults >= 20 years: 55,081  (excluded 46,235)
  Eligible for mortality linkage: 54,945  (excluded 136)
  Positive follow-up time: 52,276  (excluded 2,669)
  Valid cancer status (yes/no): 52,224  (excluded 52)
  Non-missing BMI: 51,168  (excluded 1,056)

Final analytic sample: 51,168


In [4]:
# ============================================================
# STEP 3.4 — Construct derived variables
# ============================================================

# Cancer status (binary)
df['cancer'] = (df['MCQ220'] == 1.0).astype(int)

# Diabetes: doctor-told OR HbA1c >= 6.5%
df['diabetes'] = ((df['DIQ010'] == 1.0) | (df['LBXGH'] >= 6.5)).astype(int)
# If both DIQ010 and LBXGH are missing, mark as missing
both_miss = (df['DIQ010'].isna()) & (df['LBXGH'].isna())
df.loc[both_miss, 'diabetes'] = np.nan

# Hypertension (binary)
df['hypertension'] = (df['BPQ020'] == 1.0).astype(int)
df.loc[df['BPQ020'].isna(), 'hypertension'] = np.nan

# BMI categories
df['obese'] = (df['BMXBMI'] >= 30).astype(int)
df['bmi_cat'] = pd.cut(df['BMXBMI'], 
                        bins=[0, 18.5, 25, 30, 100], 
                        labels=['Underweight', 'Normal', 'Overweight', 'Obese'],
                        right=False)

# Smoking status (Never / Former / Current)
def classify_smoking(row):
    smq020 = row.get('SMQ020', np.nan)
    smq040 = row.get('SMQ040', np.nan)
    if smq020 == 2.0:
        return 'Never'
    elif smq020 == 1.0:
        if smq040 in [1.0, 2.0]:  # every day or some days
            return 'Current'
        elif smq040 == 3.0:  # not at all
            return 'Former'
        else:
            return 'Former'  # smoked 100+ but no current info -> former
    return np.nan

df['smoking'] = df.apply(classify_smoking, axis=1)

# Outcome variables
df['dead'] = (df['MORTSTAT'] == 1.0).astype(int)
df['follow_up_yrs'] = df['PERMTH_EXM'] / 12.0

# Demographics recoding
df['female'] = (df['RIAGENDR'] == 2.0).astype(int)
df['age'] = df['RIDAGEYR']

race_map = {1.0: 'Mexican American', 2.0: 'Other Hispanic', 
            3.0: 'Non-Hispanic White', 4.0: 'Non-Hispanic Black', 5.0: 'Other/Multi'}
df['race_ethnicity'] = df['RIDRETH1'].map(race_map)

# Age groups for stratified analysis
df['age_group'] = pd.cut(df['age'], bins=[20, 40, 60, 80, 100], 
                          labels=['20-39', '40-59', '60-79', '80+'], right=False)

# Survey weights: divide by 10 for pooling 10 cycles (per NCHS guidance)
df['wt_pooled'] = df['WTMEC2YR'] / 10.0

print('=== DERIVED VARIABLES SUMMARY ===')
for var in ['cancer', 'diabetes', 'hypertension', 'obese', 'smoking', 'dead']:
    print(f'\n{var}:')
    print(df[var].value_counts(dropna=False))

=== DERIVED VARIABLES SUMMARY ===

cancer:
cancer
0    46453
1     4715
Name: count, dtype: int64

diabetes:
diabetes
0.0    43625
1.0     7543
Name: count, dtype: int64

hypertension:
hypertension
0.0    33450
1.0    17595
NaN      123
Name: count, dtype: int64

obese:
obese
0    32535
1    18633
Name: count, dtype: int64

smoking:
smoking
Never      27862
Former     12569
Current    10699
NaN           38
Name: count, dtype: int64

dead:
dead
0    43390
1     7778
Name: count, dtype: int64


In [5]:
# ============================================================
# STEP 3.5 — Validate the analytic cohort
# ============================================================

print('=== ANALYTIC COHORT SUMMARY ===')
print(f'Total participants: {len(df):,}')
print(f'Cancer survivors: {df["cancer"].sum():,} ({df["cancer"].mean()*100:.1f}%)')
print(f'Deaths: {df["dead"].sum():,} ({df["dead"].mean()*100:.1f}%)')
print()

# Cancer survivor sub-cohort
cs = df[df['cancer'] == 1]
print(f'=== CANCER SURVIVOR SUB-COHORT ===')
print(f'N: {len(cs):,}')
print(f'Deaths: {cs["dead"].sum():,} ({cs["dead"].mean()*100:.1f}%)')
print(f'Mean age: {cs["age"].mean():.1f} (SD {cs["age"].std():.1f})')
print(f'Female: {cs["female"].mean()*100:.1f}%')
print(f'Mean follow-up: {cs["follow_up_yrs"].mean():.1f} years (SD {cs["follow_up_yrs"].std():.1f})')
print(f'Median follow-up: {cs["follow_up_yrs"].median():.1f} years')
print()
print(f'Comorbidities among cancer survivors:')
print(f'  Diabetes: {cs["diabetes"].sum():,} ({cs["diabetes"].mean()*100:.1f}%)')
print(f'  Hypertension: {cs["hypertension"].sum():,} ({cs["hypertension"].mean()*100:.1f}%)')
print(f'  Obesity: {cs["obese"].sum():,} ({cs["obese"].mean()*100:.1f}%)')
print()

# Missingness in derived variables
print('=== MISSINGNESS IN DERIVED VARIABLES ===')
for var in ['cancer', 'diabetes', 'hypertension', 'obese', 'smoking', 'dead', 'follow_up_yrs', 'age', 'female', 'race_ethnicity']:
    miss = df[var].isna().sum()
    pct = df[var].isna().mean() * 100
    print(f'  {var}: {miss:,} ({pct:.1f}%)')

=== ANALYTIC COHORT SUMMARY ===
Total participants: 51,168
Cancer survivors: 4,715 (9.2%)
Deaths: 7,778 (15.2%)

=== CANCER SURVIVOR SUB-COHORT ===
N: 4,715
Deaths: 1,676 (35.5%)


Mean age: 65.7 (SD 14.4)
Female: 52.8%
Mean follow-up: 7.8 years (SD 5.1)
Median follow-up: 6.9 years

Comorbidities among cancer survivors:
  Diabetes: 1,033.0 (21.9%)
  Hypertension: 2,618.0 (55.5%)
  Obesity: 1,678 (35.6%)

=== MISSINGNESS IN DERIVED VARIABLES ===
  cancer: 0 (0.0%)
  diabetes: 0 (0.0%)
  hypertension: 123 (0.2%)
  obese: 0 (0.0%)
  smoking: 38 (0.1%)
  dead: 0 (0.0%)
  follow_up_yrs: 0 (0.0%)
  age: 0 (0.0%)
  female: 0 (0.0%)
  race_ethnicity: 0 (0.0%)


In [6]:
# ============================================================
# STEP 3.6 — Cancer survivors by cycle (post-exclusion)
# ============================================================

cycle_summary = df.groupby('cycle').agg(
    total=('cancer', 'count'),
    cancer_n=('cancer', 'sum'),
    deaths=('dead', 'sum'),
    mean_age=('age', 'mean'),
    mean_fu=('follow_up_yrs', 'mean')
).reset_index()

cycle_summary['cancer_pct'] = cycle_summary['cancer_n'] / cycle_summary['total'] * 100
cycle_summary['mortality_pct'] = cycle_summary['deaths'] / cycle_summary['total'] * 100

print('=== ANALYTIC COHORT BY CYCLE ===')
print(cycle_summary.to_string(index=False))
print(f'\nTotal cancer survivors (analytic): {cycle_summary["cancer_n"].sum():,.0f}')
print(f'Total deaths (analytic): {cycle_summary["deaths"].sum():,.0f}')

=== ANALYTIC COHORT BY CYCLE ===
    cycle  total  cancer_n  deaths  mean_age   mean_fu  cancer_pct  mortality_pct
1999-2000   4368       332    1393 49.675366 16.866739    7.600733      31.891026
2001-2002   4679       413    1178 48.184655 15.858980    8.826672      25.176320
2003-2004   4625       434    1229 50.335784 13.910631    9.383784      26.572973
2005-2006   4674       381     892 47.932606 12.716178    8.151476      19.084296
2007-2008   5593       541     995 50.431790 10.953931    9.672805      17.790095
2009-2010   5974       594     781 49.345665  9.388098    9.943087      13.073318
2011-2012   5220       451     547 48.630268  7.648356    8.639847      10.478927
2013-2014   5509       518     417 49.020149  5.793762    9.402795       7.569432
2015-2016   5388       517     234 49.384744  3.894302    9.595397       4.342984
2017-2018   5138       534     112 51.255158  1.960296   10.393149       2.179837

Total cancer survivors (analytic): 4,715
Total deaths (analytic)

In [7]:
# ============================================================
# STEP 3.7 — Select final columns and save
# ============================================================

final_cols = [
    'SEQN', 'cycle', 'age', 'age_group', 'female', 'race_ethnicity',
    'cancer', 'diabetes', 'hypertension', 'obese', 'bmi_cat', 'BMXBMI', 'LBXGH',
    'smoking', 'dead', 'follow_up_yrs', 'UCOD_LEADING',
    'wt_pooled', 'SDMVPSU', 'SDMVSTRA'
]

# Keep only columns that exist
final_cols = [c for c in final_cols if c in df.columns]
analytic = df[final_cols].copy()

# Save
analytic.to_csv('data/analytic_cohort.csv', index=False)
print(f'Saved: data/analytic_cohort.csv')
print(f'Shape: {analytic.shape[0]:,} rows x {analytic.shape[1]} cols')
print(f'\nColumn list:')
for col in analytic.columns:
    print(f'  {col}: {analytic[col].dtype}')
print(f'\nFile size: {Path("data/analytic_cohort.csv").stat().st_size / 1e6:.1f} MB')

Saved: data/analytic_cohort.csv
Shape: 51,168 rows x 19 cols

Column list:
  SEQN: int64
  cycle: object
  age: float64
  age_group: category
  female: int64
  race_ethnicity: object
  cancer: int64
  diabetes: float64
  hypertension: float64
  obese: int64
  bmi_cat: category
  BMXBMI: float64
  LBXGH: float64
  smoking: object
  dead: int64
  follow_up_yrs: float64
  wt_pooled: float64
  SDMVPSU: float64
  SDMVSTRA: float64

File size: 6.3 MB


---
## Step 3 Summary

### What was done
1. Loaded and merged all 10 NHANES cycles (1999-2018): 7 survey components + mortality per cycle
2. Applied 5 inclusion/exclusion criteria (CONSORT-style flow documented above)
3. Constructed 10 derived variables: cancer, diabetes (composite), hypertension, obesity, BMI category, smoking status, mortality, follow-up time, pooled survey weights
4. Validated cohort characteristics and missingness
5. Saved final analytic CSV

### Key Numbers (update after execution)
- Final analytic N: check output
- Cancer survivors: check output
- Deaths: check output
- Median follow-up: check output

### Decisions Made
- **Diabetes definition:** Composite of doctor-diagnosis (DIQ010==1) OR lab-confirmed (HbA1c >= 6.5%). This captures undiagnosed diabetes, strengthening the exposure definition.
- **Smoking:** 3-level (Never/Former/Current) per CDC convention. Participants who smoked 100+ cigarettes but have no current smoking data are classified as Former.
- **Survey weights:** Divided by 10 for pooling across 10 cycles per NCHS Technical Guidance on Combining Cycles.
- **BMI exclusion:** Required non-missing BMI since obesity is a primary exposure. This may slightly reduce sample but ensures complete exposure data.

### Next Step
**Step 4:** Table 1 (baseline characteristics stratified by cancer status)